# 🧪 Modelo Seq2Seq baseado em google/mt5-small

O **mt5-small** é um modelo **Seq2Seq** (encoder-decoder) baseado na arquitetura **mT5** (T5 multilingual), pré-treinado em 101 idiomas incluindo o português. Ao contrário de modelos causais, o T5 usa um encoder para processar a entrada e um decoder para gerar a saída — o que o torna naturalmente adequado para tarefas de geração condicional como sumarização, tradução e resposta a perguntas.


## 1. Requisitos e Importações

In [ ]:
!pip install transformers datasets peft accelerate torch sentencepiece

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    !pip install --upgrade torchao>=0.16.0

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
    TaskType
)
import torch

## 2. Carregar e Preparar o Dataset

Utilizaremos o arquivo `dataset.jsonl`, onde cada linha contém uma instrução (`Instruction`) e a saída desejada (`Output`).  
Em modelos Seq2Seq como o T5, a instrução e a saída são mantidas **separadas**: a instrução alimenta o **encoder** e a saída é usada como **label** para o **decoder**.  
Mantemos os campos `source` e `target` distintos para a tokenização posterior.


In [ ]:
def convert_to_hf_format(example):
    """Mantém instrução (source) e saída (target) separadas — necessário para Seq2Seq."""
    return {
        "source": f"Instruction: {example['Instruction']}",
        "target": example['Output']
    }

# Carrega o arquivo JSON Lines (campos: "Instruction" e "Output")
dataset = load_dataset('json', data_files='dataset.jsonl')
# Aplica a conversão
dataset = dataset.map(convert_to_hf_format)
# Divide em treino e validação
dataset = dataset["train"].train_test_split(test_size=0.2)
print(dataset)


## 3. Carregar o Modelo Pré-Treinado e o Tokenizador

Por ser Seq2Seq, usamos `AutoModelForSeq2SeqLM`, o tokenizador mT5 já define `pad_token` (`<pad>`) por padrão, mas verificamos mesmo assim para garantir compatibilidade.


In [ ]:
model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, low_cpu_mem_usage=True)

# mT5 já possui pad_token (<pad>), mas garantimos por segurança
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # fallback; mT5 normalmente já tem <pad>

print(f"Modelo carregado: {model_name}")


## 4. Inferência ANTES do Fine-Tuning

Antes de qualquer treinamento, vamos ver como o modelo base responde a uma pergunta presente no nosso dataset.  
Em modelos Seq2Seq, a geração é feita passando a instrução ao **encoder** (`input_ids`) e deixando o **decoder** gerar a saída — não há necessidade de separar o prompt da resposta na saída decodificada.


In [ ]:
def generate_response(model, tokenizer, instruction, input_text="", max_new_tokens=80):
    """Gera uma resposta a partir de uma instrução usando um modelo Seq2Seq (encoder-decoder)."""
    if input_text:
        prompt = f"Instruction: {instruction}\nInput: {input_text}"
    else:
        prompt = f"Instruction: {instruction}"

    # Tokeniza apenas a instrução (vai para o encoder)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=192
    )

    # O decoder gera a saída automáticamente
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
    )
    # No Seq2Seq a saída decodificada é apenas a resposta (sem o prompt)
    resposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return resposta


# Exemplo de instrução (presente no dataset.jsonl)
test_instruction = "Qual é a principal característica técnica abordada no manual para sistemas de comandos de segurança?"

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")


> **Observação:** O modelo base provavelmente gerará um texto genérico ou sem relação direta com a instrução, pois ainda não foi adaptado.

## 5. Tokenização do Dataset

Em modelos Seq2Seq, a tokenização é **bidirecional**: a instrução (`source`) é tokenizada separadamente da saída (`target`).  
Os `labels` são os `input_ids` do target; o `DataCollatorForSeq2Seq` cuida de mascará-los adequadamente durante o treinamento (tokens de padding recebem label `-100` e são ignorados no cálculo da loss).  
Usamos `max_length=192` para o encoder e `max_target_length=192` para o decoder.


In [ ]:
MAX_INPUT_LENGTH = 192   # comprimento máximo da instrução (encoder)
MAX_TARGET_LENGTH = 192  # comprimento máximo da resposta (decoder)

def tokenize_function(examples):
    # Tokeniza as instruções (encoder)
    model_inputs = tokenizer(
        examples["source"],
        padding="max_length",
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    )
    # Tokeniza os alvos (decoder / labels)
    labels = tokenizer(
        text_target=examples["target"],
        padding="max_length",
        truncation=True,
        max_length=MAX_TARGET_LENGTH
    )
    # Substitui padding nos labels por -100 para ignorar no cálculo da loss
    labels_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels_ids
    return model_inputs

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenizado:", tokenized_datasets)


## 6. Preparar o Modelo para LoRA

A função `prepare_model_for_kbit_training` ativa técnicas como *gradient checkpointing* e ajusta a arquitetura para treinamento eficiente.  
É útil tanto com, quanto sem, quantização para melhor gerenciamento de memória, e funciona igualmente para modelos Seq2Seq.


In [7]:
model = base_model
model = prepare_model_for_kbit_training(model)


## 7. Configurar e Injetar LoRA

- **r**: posto das matrizes de adaptação.
- **lora_alpha**: fator de escala $\alpha$.
- **target_modules**: os módulos onde aplicaremos LoRA. Na arquitetura **mT5 (mt5-small)**, os módulos de atenção são `q` e `v` (query e value).
- **lora_dropout**: dropout aplicado às matrizes LoRA para regularização.
- **bias**: `"none"` significa que não treinamos os vieses.
- **task_type**: como é um modelo encoder-decoder, usamos `TaskType.SEQ_2_SEQ_LM`.

Em seguida, criamos o modelo PEFT com `get_peft_model`, que insere os adaptadores LoRA e **congela** o restante do modelo.


In [ ]:
lora_config = LoraConfig(
    r=16,                           # rank da decomposição
    lora_alpha=32,                  # escala alpha
    target_modules=["q", "v"],      # projeções de atenção do mT5 (mt5-small)
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,  # encoder-decoder
    inference_mode=False            # False = modo treinamento
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


✅ **Interpretação:** Apenas uma fração mínima do total de parâmetros será atualizada.  
No exemplo, menos de 1% dos pesos são treináveis – é a essência do PEFT.

## 8. Data Collator para Seq2Seq

O `DataCollatorForSeq2Seq` é específico para modelos encoder-decoder. Ele:
- Faz o padding dinâmico de `input_ids` e `labels` dentro de cada batch.
- Substitui automaticamente tokens de padding nos `labels` por `-100`, que são ignorados no cálculo da cross-entropy loss.
- Elimina a necessidade de pré-preencher tudo em `max_length` (o que foi feito na tokenização apenas para consistência).


In [9]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,           # necessário para o collator ajustar decoder_input_ids
    label_pad_token_id=-100,  # ignora padding nos labels durante a loss
    pad_to_multiple_of=8  # eficiência em hardware compatível
)

## 9. Argumentos de Treinamento

Definimos os hiperparâmetros do treinamento.  
Embora nosso Dataset seja grande, usaremos 50 épocas e uma taxa de aprendizado de `2e-4`.  
O `predict_with_generate=False` é mantido por padrão no `Trainer` — para avaliação por geração, seria necessário usar `Seq2SeqTrainer`; aqui usamos o `Trainer` padrão com avaliação por loss.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",          # diretório de saída
    eval_strategy="steps",
    eval_steps=50,
    learning_rate=2e-4,              # LR adequado para LoRA em mT5
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,   # batch efetivo = 4
    num_train_epochs=50,             # mt5-small converge bem
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    report_to="none",                # desabilita logging para wandb/mlflow
)


## 10. Inicializar o Trainer

O `Trainer` do Hugging Face orquestra todo o ciclo de treinamento, avaliação e salvamento.

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

## 11. Treinar o Modelo

In [ ]:
trainer.train()

## 12. Salvar o Modelo Ajustado e o Tokenizador

Ao final do treinamento, salvamos os pesos LoRA (apenas os adaptadores) e o tokenizador.


In [ ]:
model.save_pretrained("lora_finetuned_model")
tokenizer.save_pretrained("mt5_tokenizer")


## 13. Inferência APÓS o Fine-Tuning

Agora carregamos o modelo ajustado e comparamos sua resposta com a versão base, usando **exatamente a mesma instrução**.  
O modelo Seq2Seq gera a resposta exclusivamente no decoder, por isso a função `generate_response` já decodifica apenas os tokens gerados.


In [ ]:
# Recarrega o modelo base (mt5-small) e o tokenizador salvos
finetuned_base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, low_cpu_mem_usage=True)
finetuned_tokenizer = AutoTokenizer.from_pretrained("mt5_tokenizer")

# Garante que o token de padding esteja definido (mT5 já tem <pad>)
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

# Aplica os adaptadores LoRA treinados sobre o modelo base
finetuned_model = PeftModel.from_pretrained(finetuned_base_model, "lora_finetuned_model")


In [ ]:
print("=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta ajustada: {generate_response(finetuned_model, finetuned_tokenizer, test_instruction)}")


## 14 Comparação e Conclusão

- **Antes do fine-tuning:** o modelo base não conhecia nosso domínio; sua resposta era genérica ou incoerente.
- **Depois do fine-tuning:** com apenas uma fração dos parâmetros treinados (via LoRA), o modelo aprendeu a estrutura desejada e gera respostas alinhadas com os exemplos fornecidos.